[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaytAlhikmah/hands-on-llms-for-swes/blob/main/chapters/4/notebook.ipynb)

# Chapter 4 Understanding Neural Networks from Scratch


## The Mystery Box

In Chapter 3, you used `TransitionLearner` - a black box that learned to predict the next character in a name. It worked just like the counting approach, but something was different inside.

```python
# Chapter 3 - this was a mystery
model = TransitionLearner(n=28)
model.train(xs, ys, steps=200)
```

This was implemented as a neural network, and to understand it 
**Our goal is to learn**
1. What is a neuron and what is a neural network
2. How gradient descent works.

**But first, we need to start simple...**

---

## Setup

In [ ]:
# Install the course package
!pip install -q git+https://github.com/BaytAlhikmah/hands-on-llms-for-swes.git#subdirectory=pkg

In [ ]:
import math
import random
import matplotlib.pyplot as plt
import numpy as np
from alhikmah_llms import Chapter4

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)

print('✓ Setup complete!')
print('\nIn this notebook, we\'ll build neural networks from scratch using only NumPy.')

---

## Part 1: The Simplest Possible Neuron

A **neuron** is the basic building block of neural networks. Let's start with the absolute simplest version.

### Exercise 1: A Neuron is Just y = w*x + b

A neuron takes an input, multiplies it by a **weight**, adds a **bias**, and returns the result.

```
output = weight × input + bias
```

That's it! Let's see it in action.

In [ ]:
# TODO: Implement the simplest neuron function
def neuron(x, w, b):
    """
    The simplest possible neuron.
    
    Args:
        x: input number
        w: weight (how much to scale the input)
        b: bias (how much to shift the output)
    
    Returns:
        output number
    """
    return w * x + b

# Test it
x = 3.0
w = 2.0
b = 1.0

output = neuron(x, w, b)
print(f"Input: {x}")
print(f"Weight: {w}, Bias: {b}")
print(f"Output: {output}")
print(f"\nCalculation: {w} × {x} + {b} = {output}")

### Exercise 2: How w and b Affect the Output

Let's visualize what happens when we change the weight and bias.

**Key concepts:**
- **Weight (w)**: Controls the SLOPE - how steep the line is
- **Bias (b)**: Controls the SHIFT - where the line crosses the y-axis

In [ ]:
# Create a range of input values
x_values = np.linspace(-5, 5, 100)

# Try different weight and bias combinations
configs = [
    (1.0, 0.0, "w=1, b=0 (no change)"),
    (2.0, 0.0, "w=2, b=0 (double the input)"),
    (1.0, 3.0, "w=1, b=3 (shift up by 3)"),
    (0.5, -2.0, "w=0.5, b=-2 (half, then shift down)")
]

# Use Chapter4 visualization method
Chapter4.visualize_neuron_variants(neuron, x_values, configs)

print("Key insights:")
print("  • Weight controls the SLOPE")
print("  • Bias controls the SHIFT")
print("  • A neuron draws a straight line!")

### Exercise 3: Multiple Inputs - Extending to 2D

So far our neuron had ONE input. What if we have TWO inputs?

```
output = w1 × x1 + w2 × x2 + b
```

This is still just a linear combination - we're adding up weighted inputs.

In [ ]:
def neuron_2d(x1, x2, w1, w2, b):
    """
    A neuron with two inputs.
    
    Args:
        x1, x2: two input numbers
        w1, w2: two weights
        b: bias
    """
    return w1 * x1 + w2 * x2 + b

# Test it
x1, x2 = 2.0, 3.0
w1, w2 = 1.0, -0.5
b = 1.0

output = neuron_2d(x1, x2, w1, w2, b)
print(f"Inputs: x1={x1}, x2={x2}")
print(f"Weights: w1={w1}, w2={w2}, bias={b}")
print(f"Output: {output}")
print(f"\nCalculation: {w1}×{x1} + {w2}×{x2} + {b} = {output}")

  **Geometric interpretation:**
  - With 1 input: neuron's decision boundary is a POINT on the number line
  - With 2 inputs: neuron's decision boundary is a LINE in 2D space
  - With 3 inputs: neuron's decision boundary is a PLANE in 3D space
  - With n inputs: neuron's decision boundary is a HYPERPLANE in n-dimensional space

This line/plane is called a **decision boundary** when we use the neuron for classification.

### Exercise 4: Making Decisions with a Neuron

Let's use our neuron to solve a simple problem: **Is a number big or small?**

**Rule:** Numbers ≥ 6 are "big" (class 1), numbers < 6 are "small" (class 0)

We'll use a **threshold**: if output > 0, predict "big", otherwise predict "small".

In [ ]:
# Our dataset: numbers and their labels
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
labels = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]  # 0 = small, 1 = big

print("Dataset:")
for num, label in zip(numbers, labels):
    print(f"  {num} → {label} {'(small)' if label == 0 else '(big)'}")

print("\n" + "="*70)
print("KEY QUESTION: How does a LINE become a POINT?")
print("="*70)
print("\nThe neuron computes: y = w×x + b  (this is a LINE in 2D space)")
print("For classification, we use a threshold: if y ≥ 0, predict 'big'")
print("\nDecision boundary = where y = 0")
print("Solving: w×x + b = 0  →  x = -b/w  (this is a POINT on the number line!)")
print("\nLet's visualize this transformation...")
print("="*70)

In [ ]:
def predict(x, w, b):
    """
    Use neuron to predict if a number is big or small.
    
    Returns:
        0 if output < 0 (small)
        1 if output >= 0 (big)
    """
    output = neuron(x, w, b)
    return 1 if output >= 0 else 0

# Example: Let's see how y = wx + b becomes a decision point
w_example = 1.0
b_example = -6.0

print(f"Example with w={w_example}, b={b_example}:")
print(f"\nThe neuron equation: y = {w_example}×x + {b_example}")
print(f"Decision boundary where y = 0:")
print(f"  {w_example}×x + {b_example} = 0")
print(f"  x = {-b_example/w_example:.2f}")
print(f"\nSo the decision point is at x = {-b_example/w_example:.2f}")
print("\n" + "="*70)

# Visualize the transformation from line to point
Chapter4.visualize_line_to_decision_boundary(w_example, b_example, numbers, labels)

print("\n" + "="*70)
print("Now let's try different weights and see how the decision boundary moves:")
print("="*70)

In [ ]:
# Try different weights and biases to find good decision boundaries
attempts = [
    (1.0, -3.0, "First try"),
    (1.0, -5.0, "Getting closer"),
    (1.0, -6.0, "Even closer"),
]

# Visualize all attempts
Chapter4.visualize_decision_attempts(numbers, labels, attempts)

print("\n" + "="*50)
print("Notice how changing b moves the decision boundary!")
print("  • b = -3  →  boundary at x = 3")
print("  • b = -5  →  boundary at x = 5")
print("  • b = -6  →  boundary at x = 6 (perfect!)")
print("\nFinding good weights manually is tedious!")
print("We need a way to find them AUTOMATICALLY...")
print("="*50)

### Exercise 5: Activation Functions - Why We Need Them

Our current neuron outputs any number from -∞ to +∞. For classification, we want **probabilities** (numbers between 0 and 1).

An **activation function** transforms the neuron's output. Let's learn two important ones:

1. **Sigmoid**: Squashes output to (0, 1) - good for probabilities
   
   $$\sigma(z) = \frac{1}{1 + e^{-z}}$$

2. **ReLU**: Sets negative values to 0 - good for hidden layers
   
   $$\text{ReLU}(z) = \max(0, z)$$

In [ ]:
def sigmoid(z):
    """
    Sigmoid activation: 1 / (1 + e^(-z))
    
    Squashes any input to range (0, 1)
    Output can be interpreted as probability!
    """
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    """
    ReLU activation: max(0, z)
    
    If z is negative, return 0.
    If z is positive, return z.
    """
    return np.maximum(0, z) if isinstance(z, np.ndarray) else max(0, z)

# Test sigmoid
print("Sigmoid examples:")
for z in [-5, -1, 0, 1, 5]:
    print(f"  sigmoid({z:2d}) = {sigmoid(z):.4f}")

print("\nReLU examples:")
for z in [-5, -1, 0, 1, 5]:
    print(f"  relu({z:2d}) = {relu(z):.4f}")

In [ ]:
# Visualize both activation functions
z_values = np.linspace(-10, 10, 200)

# Use Chapter4 visualization method
Chapter4.visualize_activation_functions(sigmoid, relu, z_values)

print("\n" + "="*70)
print("Now let's see how sigmoid works on our actual problem!")
print("="*70)
print("\nRecall our 'big vs small' classification problem.")
print("Let's use w=1.0, b=-6.0 and see the THREE steps:")
print("  1. Compute linear output: y = wx + b")
print("  2. Apply sigmoid: probability = σ(y)")
print("  3. Classify: if probability ≥ 0.5, predict 'big'")
print("="*70)

# Show how sigmoid transforms the neuron output for our problem
w_demo = 1.0
b_demo = -6.0
Chapter4.visualize_sigmoid_on_problem(w_demo, b_demo, numbers, labels, neuron, sigmoid)

print("\n" + "="*70)
print("KEY INSIGHTS:")
print("="*70)
print("\n1. LINEAR OUTPUT (y = wx + b):")
print("   • Crosses y=0 at x=6")
print("   • Negative values → will become probabilities < 0.5")
print("   • Positive values → will become probabilities > 0.5")
print("\n2. AFTER SIGMOID:")
print("   • S-shaped curve (smooth transition)")
print("   • Crosses p=0.5 at exactly x=6 (where y was 0!)")
print("   • Output is now a PROBABILITY between 0 and 1")
print("\n3. CLASSIFICATION:")
print("   • Numbers < 6: probability < 0.5 → predict 'small'")
print("   • Numbers ≥ 6: probability ≥ 0.5 → predict 'big'")
print("\n💡 The sigmoid converts the linear decision boundary (y=0)")
print("   into a smooth probability curve (p=0.5)!")
print("="*70)

---

## Part 2: Learning Weights Automatically

Manual trial-and-error doesn't scale. We need the computer to find good weights **automatically**.

This is where **gradient descent** comes in - the fundamental algorithm that powers all neural network training.

### Exercise 6: How Do We Measure "Wrong"?

To improve our weights, we first need to measure how wrong our predictions are.

This is called a **loss function**. We'll use **Binary Cross-Entropy** (BCE) - the same cross-entropy we learned in Chapter 2!

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    """
    Binary Cross-Entropy loss (from Chapter 2!).
    
    BCE = -[y*log(p) + (1-y)*log(1-p)]
    
    Args:
        y_true: true label (0 or 1)
        y_pred: predicted probability (between 0 and 1)
    
    Returns:
        loss (lower is better)
    """
    epsilon = 1e-8  # Small number to avoid log(0)

    # what clip does is 1 -> 1-eps and 0 -> eps for the entire array
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Test it
print("Binary Cross-Entropy Examples:")
print("\nWhen true label is 1:")
print(f"  Predicted 0.9 → Loss = {binary_cross_entropy(1, 0.9):.4f}  (good prediction, low loss)")
print(f"  Predicted 0.5 → Loss = {binary_cross_entropy(1, 0.5):.4f}  (uncertain, medium loss)")
print(f"  Predicted 0.1 → Loss = {binary_cross_entropy(1, 0.1):.4f}  (bad prediction, high loss)")

print("\nWhen true label is 0:")
print(f"  Predicted 0.1 → Loss = {binary_cross_entropy(0, 0.1):.4f}  (good prediction, low loss)")
print(f"  Predicted 0.5 → Loss = {binary_cross_entropy(0, 0.5):.4f}  (uncertain, medium loss)")
print(f"  Predicted 0.9 → Loss = {binary_cross_entropy(0, 0.9):.4f}  (bad prediction, high loss)")

print("\n" + "="*60)
print("KEY CONNECTION TO CHAPTER 2:")
print("="*60)
print("Binary Cross-Entropy is the SAME cross-entropy we learned!")
print("  • It measures how wrong our probability distribution is")
print("  • Lower loss = better predictions")
print("  • This is the standard loss for binary classification")
print("="*60)

In [ ]:
def compute_loss(w, b, numbers, labels):
    """
    Compute average BCE loss over all examples.
    """
    total_loss = 0
    for x, y_true in zip(numbers, labels):
        # Forward pass
        z = neuron(x, w, b)
        y_pred = sigmoid(z)
        
        # Compute loss
        loss = binary_cross_entropy(y_true, y_pred)
        total_loss += loss
    
    return total_loss / len(numbers)

# Try different weights and see their loss
print("Loss for different weights:\n")
for w, b in [(1.0, -3.0), (1.0, -5.0), (1.0, -6.0), (1.0, -7.0)]:
    loss = compute_loss(w, b, numbers, labels)
    print(f"  w={w}, b={b:>4} → Loss = {loss:.4f}")

print("\nLower loss = better weights!")
print("But how do we find the best weights automatically?")

### Exercise 7: The Loss Landscape - Finding the Valley

Let's visualize how loss changes as we vary our weights. This is called the **loss landscape**.

The best weights are at the **bottom of the valley** - where loss is lowest.

In [ ]:
# ========== PART 1: Constrained Optimization (b fixed) ==========
print("="*70)
print("PART 1: CONSTRAINED OPTIMIZATION (b = -5.0 fixed)")
print("="*70)

# Fix bias, vary weight
b_fixed = -5.0
weight_values = np.linspace(-2, 4, 200)
losses = [compute_loss(w, b_fixed, numbers, labels) for w in weight_values]

# Visualize 1D loss landscape
Chapter4.visualize_loss_landscape(weight_values, losses, b_fixed)

best_idx = np.argmin(losses)
w_constrained = weight_values[best_idx]
loss_constrained = losses[best_idx]

print(f"\nBest with b=-5.0 fixed: w={w_constrained:.3f}, loss={loss_constrained:.4f}")

# ========== PART 2: Global Optimization (both w and b free) ==========
print("\n" + "="*70)
print("PART 2: GLOBAL OPTIMIZATION (both w and b free)")
print("="*70)

# Grid search over expanded range
w_values_2d = np.linspace(0, 50, 200)
b_values_2d = np.linspace(-18, -300, 200)

best_loss_global = float('inf')
w_global = None
b_global = None

print("\nSearching 200×200 = 40,000 points...")
for w in w_values_2d:
    for b in b_values_2d:
        loss = compute_loss(w, b, numbers, labels)
        if loss < best_loss_global:
            best_loss_global = loss
            w_global = w
            b_global = b

print(f"Global optimum: w={w_global:.3f}, b={b_global:.3f}, loss={best_loss_global:.6f}")

# ========== PART 3: 2D Loss Landscape ==========
print("\n" + "="*70)
print("2D LOSS LANDSCAPE")
print("="*70)

def loss_for_viz(w, b):
    return compute_loss(w, b, numbers, labels)

Chapter4.visualize_2d_loss_landscape(
    w_range=(0.5, 5.0, 100),
    b_range=(-18, -3, 100),
    compute_loss_func=loss_for_viz
)

# ========== PART 4: Loss Values at Different Points ==========
print("\n" + "="*70)
print("LOSS VALUES AT DIFFERENT (w, b) POINTS")
print("="*70)

# Show interesting points
interesting_points = [
    (0.5, -5.0, "Small w, moderate b"),
    (1.0, -5.0, "w=1, b=-5 (constrained line)"),
    (w_constrained, b_fixed, "Constrained optimum"),
    (1.5, -8.0, "Moderate w, more negative b"),
    (2.0, -11.0, "Larger w, matched b"),
    (3.0, -16.5, "Large w, very negative b"),
    (w_global, b_global, "Global optimum (in range)"),
]

print(f"\n{'w':<8} {'b':<10} {'Loss':<15} {'Boundary':<12} {'Description'}")
print("-"*75)

for w, b, desc in interesting_points:
    loss = compute_loss(w, b, numbers, labels)
    boundary = -b / w
    print(f"{w:<8.3f} {b:<10.2f} {loss:<15.8f} {boundary:<12.3f} {desc}")

# ========== PART 5: Sigmoid Curves at Different Points ==========
print("\n" + "="*70)
print("HOW SIGMOID CHANGES AT DIFFERENT POINTS")
print("="*70)
print("\nLet's see how the sigmoid curve changes as we move through the loss landscape:")

# Select 4 representative points
sigmoid_points = [
    (w_constrained, b_fixed, loss_constrained, "Constrained\n(b=-5 fixed)", "orange"),
    (1.5, -8.0, compute_loss(1.5, -8.0, numbers, labels), "Medium w\n(steeper)", "blue"),
    (2.5, -13.75, compute_loss(2.5, -13.75, numbers, labels), "Larger w\n(even steeper)", "purple"),
    (w_global, b_global, best_loss_global, "Global Optimum\n(steepest)", "red"),
]

Chapter4.visualize_sigmoids_at_different_points(sigmoid_points, numbers, labels, neuron, sigmoid)

# ========== PART 6: Comparison Summary ==========
print("\n" + "="*70)
print("COMPARISON: Constrained vs Global Optimum")
print("="*70)

Chapter4.compare_constrained_vs_global(
    w_constrained, b_fixed, loss_constrained,
    w_global, b_global, best_loss_global,
    numbers, labels, neuron, sigmoid
)

# ========== PART 7: Key Takeaways ==========
print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)

improvement = (loss_constrained - best_loss_global) / loss_constrained * 100

print(f"\n1. CONSTRAINED OPTIMIZATION (b=-5.0 fixed):")
print(f"   • Best w: {w_constrained:.3f}")
print(f"   • Loss: {loss_constrained:.6f}")
print(f"   • Decision boundary: x = {-b_fixed/w_constrained:.2f}")
print(f"   • Easier to visualize (1D curve)")

print(f"\n2. GLOBAL OPTIMIZATION (both w, b free):")
print(f"   • Best w: {w_global:.3f}, b: {b_global:.3f}")
print(f"   • Loss: {best_loss_global:.6f}")
print(f"   • Decision boundary: x = {-b_global/w_global:.2f}")
print(f"   • {improvement:.1f}% better than constrained!")

print(f"\n3. WHY GLOBAL IS BETTER:")
print(f"   • Larger w ({w_global:.2f} vs {w_constrained:.2f}) = steeper sigmoid")
print(f"   • Steeper sigmoid = more confident predictions")
print(f"   • Predictions closer to 0 or 1 = lower cross-entropy loss")

print(f"\n4. THE PATTERN:")
print(f"   • As w increases, loss decreases")
print(f"   • We adjust b to keep boundary near x=5.5")
print(f"   • The sigmoid gets steeper and more confident")

print("\n5. NEXT: GRADIENT DESCENT")
print("   • Will automatically find the optimum")
print("   • By moving in both w and b simultaneously")
print("   • Walking 'downhill' on the loss surface")

print("="*70)

### Exercise 8: Understanding Gradients - Which Way is Downhill?

The **gradient** tells us which direction is downhill on the loss curve.

**Gradient = slope of the loss curve**

- If gradient is **positive** → loss increases to the right → go **LEFT**
- If gradient is **negative** → loss decreases to the right → go **RIGHT**

**Update rule:**
```
w_new = w_old - learning_rate × gradient
```

The negative sign ensures we go downhill!

In [ ]:
def compute_gradient(w, b, numbers, labels, h=1e-4):
    """
    Compute gradient of loss with respect to w and b.
    
    Uses numerical differentiation (finite differences):
    gradient ≈ [f(x + h) - f(x - h)] / (2h)
    """
    # Gradient with respect to w
    loss_plus = compute_loss(w + h, b, numbers, labels)
    loss_minus = compute_loss(w - h, b, numbers, labels)
    grad_w = (loss_plus - loss_minus) / (2 * h)
    
    # Gradient with respect to b
    loss_plus = compute_loss(w, b + h, numbers, labels)
    loss_minus = compute_loss(w, b - h, numbers, labels)
    grad_b = (loss_plus - loss_minus) / (2 * h)
    
    return grad_w, grad_b

# Test at a specific point
w_test = 0.5
b_test = -5.0
grad_w, grad_b = compute_gradient(w_test, b_test, numbers, labels)

print(f"At w={w_test}, b={b_test}:")
print(f"  Gradient w.r.t. w: {grad_w:.4f}")
print(f"  Gradient w.r.t. b: {grad_b:.4f}")
print("\nWhat does this mean?")
if grad_w > 0:
    print("  • Positive gradient → loss increases as w increases")
    print("  • So we should DECREASE w (move left)")
else:
    print("  • Negative gradient → loss decreases as w increases")
    print("  • So we should INCREASE w (move right)")

In [ ]:
# Visualize gradient as tangent line

w_current = 0.5
b_current = -5.0
loss_current = compute_loss(w_current, b_current, numbers, labels)
grad_w_current, _ = compute_gradient(w_current, b_current, numbers, labels)

# Use Chapter4 visualization method
Chapter4.visualize_gradient_tangent(weight_values, losses, w_current, loss_current, grad_w_current)

print("\n" + "="*70)
print("KEY INSIGHT: Gradient tells us which way is downhill!")
print("="*70)
print("\nUpdate rule: w_new = w_old - learning_rate × gradient")
print("\nThis is called GRADIENT DESCENT - the foundation of all neural network training!")

### Exercise 9: Implement Gradient Descent

Now let's put it all together: start with random weights, compute gradients, update weights, repeat.

**The Algorithm:**
```
1. Initialize w and b randomly
2. Repeat:
   a. Compute loss
   b. Compute gradients
   c. Update: w = w - lr × grad_w
              b = b - lr × grad_b
3. Until loss stops decreasing
```

In [ ]:
# Initialize weights
w = 0.5  # Starting point
b = -3.0
learning_rate = 0.4
num_steps = 200

# Track history for visualization
w_history = [w]
b_history = [b]
loss_history = [compute_loss(w, b, numbers, labels)]

print("Training with Gradient Descent...\n")
print(f"{'Step':<6} {'Weight':<10} {'Bias':<10} {'Loss':<12} {'Grad_w':<10} {'Grad_b':<10}")
print("="*70)

for step in range(num_steps):
    # Compute gradients
    grad_w, grad_b = compute_gradient(w, b, numbers, labels)
    
    # Update weights
    w = w - learning_rate * grad_w
    b = b - learning_rate * grad_b
    
    # Compute new loss
    loss = compute_loss(w, b, numbers, labels)
    
    # Track history
    w_history.append(w)
    b_history.append(b)
    loss_history.append(loss)
    
    # Print every 10 steps
    if step % 10 == 0 or step == num_steps - 1:
        print(f"{step:<6} {w:<10.4f} {b:<10.4f} {loss:<12.6f} {grad_w:<10.4f} {grad_b:<10.4f}")

print("\n" + "="*70)
print(f"✓ Training complete!")
print(f"Final weight: {w:.4f}")
print(f"Final bias: {b:.4f}")
print(f"Final loss: {loss_history[-1]:.6f}")
print(f"Initial loss: {loss_history[0]:.6f}")
print(f"Reduction: {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")
print("="*70)

In [ ]:
# Visualize the training process
Chapter4.visualize_training_progress(loss_history)

print("\nNotice:")
print("  • Loss steadily decreases")
print("  • Starts high, ends low")
print("  • The algorithm found good weights AUTOMATICALLY!")

In [ ]:
# Test the learned weights
print("Testing learned weights:\n")
print(f"{'Number':<10} {'True Label':<12} {'Prediction':<12} {'Probability':<12} {'Correct?'}")
print("="*60)

correct = 0
for num, true_label in zip(numbers, labels):
    z = neuron(num, w, b)
    prob = sigmoid(z)
    pred = 1 if prob >= 0.5 else 0
    match = "✓" if pred == true_label else "✗"
    correct += (pred == true_label)
    print(f"{num:<10} {true_label:<12} {pred:<12} {prob:<12.4f} {match}")

accuracy = correct / len(numbers) * 100
print("\n" + "="*60)
print(f"Accuracy: {correct}/{len(numbers)} = {accuracy:.0f}%")
print("="*60)
print("\n🎉 The neuron learned to classify automatically!")
print("\nWe didn't tell it what weights to use - it LEARNED them from data.")

### Visualize the Decision Boundary

Let's see what the neuron actually learned - where did it draw the line between "small" and "big"?

In [ ]:
# Visualize the learned decision boundary

# Decision boundary is where sigmoid(w*x + b) = 0.5
# This occurs when w*x + b = 0, so x = -b/w
boundary_x = -b / w

# Use Chapter4 visualization method
Chapter4.visualize_single_neuron_decision_boundary(numbers, labels, w, b, neuron, sigmoid)

print("\n" + "="*70)
print("DECISION BOUNDARY VISUALIZATION")
print("="*70)
print(f"\nThe neuron learned:")
print(f"  • Weight (w) = {w:.4f}")
print(f"  • Bias (b) = {b:.4f}")
print(f"  • Decision boundary at x = -b/w = {boundary_x:.2f}")
print(f"\nThis means:")
print(f"  • Numbers < {boundary_x:.2f} → predict 'small' (class 0)")
print(f"  • Numbers ≥ {boundary_x:.2f} → predict 'big' (class 1)")
print(f"\n✓ The neuron learned to draw a LINE that separates the classes!")
print("="*70)

---

## Part 2.5: Bridging from 1D to 2D

So far, we've been working with **1D data** - just a single input number. Our neuron learned to draw a **decision point** on the number line.

But what if we have **two features**? Our neuron equation becomes:

$$y = w_1 \cdot x_1 + w_2 \cdot x_2 + b$$

Now the decision boundary is a **line in 2D space** (not just a point!).

### Exercise 10: A Problem That Shows Why 2D is More Powerful

Let's create a dataset that demonstrates something important:
- ✗ **Using x1 alone**: Can't separate the classes
- ✗ **Using x2 alone**: Can't separate the classes  
- ✓ **Using both x1 AND x2**: A diagonal line separates them perfectly!

This shows that combining features gives us more power than using them individually.

In [ ]:
# Create 2D dataset where classes are separated by sum (x1 + x2 >= 6)
data_points = [
    (1, 1, 0),  # sum = 2
    (1, 2, 0),  # sum = 3
    (2, 2, 0),  # sum = 4
    (2, 3, 0),  # sum = 5
    (1, 4, 0),  # sum = 5
    (3, 3, 1),  # sum = 6
    (2, 4, 1),  # sum = 6
    (3, 4, 1),  # sum = 7
    (4, 3, 1),  # sum = 7
    (4, 4, 1),  # sum = 8
]

# Extract into arrays
inputs_2d = np.array([[x1, x2] for x1, x2, _ in data_points])
labels_2d = np.array([label for _, _, label in data_points])

print("2D Dataset (labeled by sum):")
print(f"{'Point':<8} {'x1':<6} {'x2':<6} {'Sum':<8} {'Label':<8} {'Class'}")
print("-" * 50)
for i, ((x1, x2), label) in enumerate(zip(inputs_2d, labels_2d), 1):
    sum_val = x1 + x2
    class_name = "0 (sum<6)" if label == 0 else "1 (sum≥6)"
    print(f"{i:<8} {x1:<6.0f} {x2:<6.0f} {sum_val:<8.0f} {label:<8} {class_name}")

print(f"\n{'='*50}")
print("KEY OBSERVATION:")
print(f"{'='*50}")
print(f"x1 alone - Class 0: {set(inputs_2d[labels_2d==0, 0])}")
print(f"           Class 1: {set(inputs_2d[labels_2d==1, 0])}")
print(f"           → OVERLAP at x1=2! Can't separate in 1D")
print()
print(f"x2 alone - Class 0: {set(inputs_2d[labels_2d==0, 1])}")
print(f"           Class 1: {set(inputs_2d[labels_2d==1, 1])}")
print(f"           → OVERLAP at x2={{3,4}}! Can't separate in 1D")
print()
print(f"But in 2D: A diagonal line x1 + x2 = 5.5 separates perfectly!")
print(f"{'='*50}")

# Visualize the 2D dataset
Chapter4.visualize_2d_dataset(inputs_2d, labels_2d, feature_names=('x1', 'x2'))

### From 1D to 2D: What Changes?

| Aspect | 1D Neuron | 2D Neuron |
|--------|-----------|-----------|
| **Input** | Single number $x$ | Two numbers $(x_1, x_2)$ |
| **Parameters** | $w, b$ (2 params) | $w_1, w_2, b$ (3 params) |
| **Equation** | $y = w \cdot x + b$ | $y = w_1 \cdot x_1 + w_2 \cdot x_2 + b$ |
| **Decision Boundary** | Point: $x = -\frac{b}{w}$ | Line: $w_1 x_1 + w_2 x_2 + b = 0$ |
| **Separates** | Number line into 2 regions | 2D plane into 2 regions |

We already defined `neuron_2d()` earlier - now let's use it with gradient descent!

In [ ]:
# Loss computation for 2D inputs
def compute_loss_2d(w1, w2, b, inputs_2d, labels):
    """Compute average BCE loss for 2D inputs."""
    total_loss = 0
    for (x1, x2), y_true in zip(inputs_2d, labels):
        z = neuron_2d(x1, x2, w1, w2, b)
        y_pred = sigmoid(z)
        loss = binary_cross_entropy(y_true, y_pred)
        total_loss += loss
    return total_loss / len(labels)

# Gradient computation for 2D neuron (3 parameters!)
def compute_gradients_2d(w1, w2, b, inputs_2d, labels, h=1e-4):
    """Compute gradients for w1, w2, and b."""
    loss = compute_loss_2d(w1, w2, b, inputs_2d, labels)
    
    # Gradient w.r.t. w1
    loss_w1_plus = compute_loss_2d(w1 + h, w2, b, inputs_2d, labels)
    grad_w1 = (loss_w1_plus - loss) / h
    
    # Gradient w.r.t. w2
    loss_w2_plus = compute_loss_2d(w1, w2 + h, b, inputs_2d, labels)
    grad_w2 = (loss_w2_plus - loss) / h
    
    # Gradient w.r.t. b
    loss_b_plus = compute_loss_2d(w1, w2, b + h, inputs_2d, labels)
    grad_b = (loss_b_plus - loss) / h
    
    return grad_w1, grad_w2, grad_b

print("✓ 2D loss and gradient functions ready")
print("\nNow we need to update THREE parameters instead of two!")

In [ ]:
# Train 2D neuron with gradient descent
w1 = np.random.randn() * 0.1
w2 = np.random.randn() * 0.1
b = np.random.randn() * 0.1

learning_rate = 0.5
num_steps = 500

loss_history_2d = []
w1_history = [w1]
w2_history = [w2]
b_history_2d = [b]

print("Training 2D neuron with gradient descent...\n")
print(f"{'Step':<6} {'w1':<10} {'w2':<10} {'b':<10} {'Loss':<12}")
print("="*55)

for step in range(num_steps):
    # Compute loss
    loss = compute_loss_2d(w1, w2, b, inputs_2d, labels_2d)
    loss_history_2d.append(loss)
    
    # Compute gradients for all 3 parameters
    grad_w1, grad_w2, grad_b = compute_gradients_2d(w1, w2, b, inputs_2d, labels_2d)
    
    # Update all 3 parameters simultaneously
    w1 = w1 - learning_rate * grad_w1
    w2 = w2 - learning_rate * grad_w2
    b = b - learning_rate * grad_b
    
    w1_history.append(w1)
    w2_history.append(w2)
    b_history_2d.append(b)
    
    if step % 50 == 0 or step == num_steps - 1:
        print(f"{step:<6} {w1:<10.4f} {w2:<10.4f} {b:<10.4f} {loss:<12.6f}")

print("\n" + "="*55)
print(f"✓ Training complete!")
print(f"Final: w1={w1:.4f}, w2={w2:.4f}, b={b:.4f}")
print(f"Final loss: {loss_history_2d[-1]:.6f}")
print("="*55)

In [ ]:
# Visualize training progress
Chapter4.visualize_gradient_descent_2d(
    inputs_2d, labels_2d,
    w1_history, w2_history, b_history_2d, loss_history_2d,
    compute_loss_func=lambda w1, w2, b: compute_loss_2d(w1, w2, b, inputs_2d, labels_2d)
)

print("\nNotice:")
print("  • Loss steadily decreased")
print("  • All 3 parameters (w1, w2, b) evolved together")
print("  • The algorithm found the optimal weights automatically!")

In [ ]:
# Check predictions
print("Predictions on 2D data:\n")
print(f"{'Point':<8} {'x1':<6} {'x2':<6} {'Sum':<6} {'Prob':<10} {'Pred':<6} {'True':<6} {'Correct?'}")
print("="*65)

correct = 0
for i, ((x1, x2), label) in enumerate(zip(inputs_2d, labels_2d), 1):
    y = neuron_2d(x1, x2, w1, w2, b)
    prob = sigmoid(y)
    pred = 1 if prob >= 0.5 else 0
    match = "✓" if pred == label else "✗"
    correct += (pred == label)
    sum_val = x1 + x2
    print(f"{i:<8} {x1:<6.0f} {x2:<6.0f} {sum_val:<6.0f} {prob:<10.4f} {pred:<6} {label:<6} {match}")

accuracy = correct / len(inputs_2d) * 100
print("\n" + "="*65)
print(f"Accuracy: {correct}/{len(inputs_2d)} = {accuracy:.0f}%")
print("="*65)

if accuracy == 100:
    print("\n🎉 Perfect! The 2D neuron learned to separate the classes!")
    print("\nKey insight:")
    print("  • Neither x1 nor x2 alone could separate the data")
    print("  • But using BOTH together, a diagonal line works perfectly")
    print("  • This is the power of combining features!")

In [ ]:
# Visualize the learned decision boundary
Chapter4.visualize_2d_learned_boundary(
    inputs_2d, labels_2d,
    w1, w2, b,
    neuron_func=neuron_2d,
    sigmoid_func=sigmoid,
    feature_names=('x1', 'x2')
)

print("\n" + "="*70)
print("THE DECISION LINE")
print("="*70)
print(f"\nThe neuron learned: {w1:.4f}·x1 + {w2:.4f}·x2 + {b:.4f} = 0")
print(f"\nThis is a LINE in 2D space that separates:")
print(f"  • Class 0 (red): points where x1 + x2 < 6")
print(f"  • Class 1 (blue): points where x1 + x2 ≥ 6")
print(f"\n✓ The neuron discovered the diagonal pattern automatically!")
print("="*70)

### Summary: The Power of 2D

**What we just showed:**

1. **The dataset was NOT linearly separable in 1D**:
   - Using x1 alone: both classes span x1 ∈ {1, 2, 3, 4}
   - Using x2 alone: both classes span x2 ∈ {1, 2, 3, 4}
   - No single threshold on either axis separates them!

2. **But it WAS linearly separable in 2D**:
   - A diagonal line `w1·x1 + w2·x2 + b = 0` perfectly separates
   - The neuron learned this line automatically through gradient descent
   - Decision boundary roughly follows `x1 + x2 = 5.5`

3. **This demonstrates a key principle**:
   - More dimensions = more power
   - Features that overlap individually can be separated when combined
   - Neural networks discover these combinations automatically!

**But what if even a 2D line isn't enough?** 🤔

That's exactly what we're about to see with the XOR problem...

---

## Part 3: When One Neuron Isn't Enough

Our single neuron can learn linear patterns (straight lines). But what about problems that need **curved** decision boundaries?

### Exercise 11: The XOR Problem

**XOR (exclusive OR):** Output 1 if inputs are different, 0 if they're the same.

```
(0, 0) → 0
(0, 1) → 1
(1, 0) → 1
(1, 1) → 0
```

Can a single neuron learn this?

In [ ]:
# XOR dataset
xor_inputs = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

xor_labels = np.array([0, 1, 1, 0])

print("XOR Truth Table:")
print(f"{'x1':<5} {'x2':<5} {'Output'}")
print("="*20)
for (x1, x2), label in zip(xor_inputs, xor_labels):
    print(f"{x1:<5} {x2:<5} {label}")

In [ ]:
# Visualize XOR problem
Chapter4.visualize_xor_problem(xor_inputs, xor_labels)

print("\nTry to draw a straight line that separates:")
print("  • Red points (0,0) and (1,1) on one side")
print("  • Blue points (0,1) and (1,0) on the other side")
print("\n...You can't! They're at opposite corners.")
print("\n" + "="*60)
print("A single neuron draws ONE line - not enough for XOR!")
print("We need MULTIPLE neurons working together.")
print("="*60)

### Can We Draw Lines to Solve XOR?

Let's try drawing different straight lines to see if any can separate the classes.

Remember: We need to separate red (0,0) and (1,1) from blue (0,1) and (1,0).

In [ ]:
# Try three different lines to separate XOR classes
Chapter4.visualize_xor_line_attempts(xor_inputs, xor_labels)

print("\n" + "="*70)
print("WHY SINGLE LINES FAIL FOR XOR")
print("="*70)
print("\nLet's check each line:")
print("\n1. HORIZONTAL LINE (x2 = 0.5):")
print("   • Red side: (0,0) ✓ and (1,0) ✗")
print("   • Blue side: (0,1) ✓ and (1,1) ✗")
print("   → FAILS: Can't separate by x2 alone")

print("\n2. VERTICAL LINE (x1 = 0.5):")
print("   • Red side: (0,0) ✓ and (0,1) ✗")
print("   • Blue side: (1,0) ✓ and (1,1) ✗")
print("   → FAILS: Can't separate by x1 alone")

print("\n3. DIAGONAL LINE (x1 + x2 = 1):")
print("   • Below line: (0,0) ✓ and (0,1) ✗")
print("   • Above line: (1,0) ✗ and (1,1) ✓")
print("   → FAILS: Red points on opposite sides!")

print("\n" + "="*70)
print("WHAT ABOUT WITH SIGMOID?")
print("="*70)
print("\nLet's apply sigmoid activation to these lines and see the probabilities:")
print("="*70)

# Show sigmoid probabilities for each line
Chapter4.visualize_xor_sigmoid_attempts(xor_inputs, xor_labels, sigmoid)

print("\n" + "="*70)
print("SIGMOID DOESN'T HELP!")
print("="*70)
print("\nEven with sigmoid activation:")
print("  • Horizontal line: Still misclassifies (1,0) and (1,1)")
print("  • Vertical line: Still misclassifies (0,1) and (1,1)")
print("  • Diagonal line: Red points still on opposite sides!")
print("\n💡 KEY INSIGHT:")
print("   Sigmoid converts linear output to probabilities (0 to 1)")
print("   But it doesn't change the DECISION BOUNDARY")
print("   The boundary is still a straight line!")
print("\n   NO SINGLE LINE can separate the XOR pattern!")
print("   Red points (0,0) and (1,1) are at OPPOSITE CORNERS.")
print("   We need a CURVED boundary = MULTIPLE lines working together!")
print("="*70)

### But What If We Had Better Features?

We just saw that **no single line** can separate XOR in the original (x1, x2) space.

But here's a key insight: **What if we transformed the features first?**

Instead of using (x1, x2) directly, what if we computed:
- **Feature 1**: x1 AND x2 (both are 1)
- **Feature 2**: x1 OR x2 (at least one is 1)

Let's see what happens...

In [ ]:
# Transform XOR inputs to (AND, OR) features
xor_transformed = np.array([
    [int(x1 and x2), int(x1 or x2)] for x1, x2 in xor_inputs
])

print("Feature Transformation:")
print(f"\n{'Original':<15} {'Transformed':<20} {'Label'}")
print(f"{'(x1, x2)':<15} {'(AND, OR)':<20} {''}")
print("-" * 50)
for (x1, x2), (and_val, or_val), label in zip(xor_inputs, xor_transformed, xor_labels):
    print(f"({x1}, {x2}){' '*10} ({and_val}, {or_val}){' '*15} {label}")

print("\n" + "="*70)
print("NOTICE SOMETHING INTERESTING?")
print("="*70)
print("\nIn the transformed space:")
print("  • Class 0: (0, 0) and (1, 1)")
print("  • Class 1: (0, 1) - both (0,1) and (1,0) map here!")
print("\nOnly 3 unique points instead of 4!")
print("And they might be separable by a line...")
print("="*70)

In [ ]:
# Visualize both spaces side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Original XOR (not separable)
ax1 = axes[0]
for (x1, x2), label in zip(xor_inputs, xor_labels):
    color = 'red' if label == 0 else 'blue'
    marker = 'o' if label == 0 else 's'
    ax1.scatter(x1, x2, c=color, s=500, marker=marker, alpha=0.7, 
               edgecolors='k', linewidth=3, zorder=5)
    ax1.text(x1, x2, f'({x1},{x2})', ha='center', va='center', 
            fontsize=11, fontweight='bold', color='white')

ax1.set_xlabel('x1', fontsize=14, fontweight='bold')
ax1.set_ylabel('x2', fontsize=14, fontweight='bold')
ax1.set_title('Original Features (x1, x2)\n❌ NOT Linearly Separable', 
             fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-0.3, 1.3)
ax1.set_ylim(-0.3, 1.3)
ax1.set_aspect('equal')

# Right: Transformed features (separable!)
ax2 = axes[1]

# Draw decision line: OR = AND + 0.5
and_line = np.linspace(-0.3, 1.3, 100)
or_line = and_line + 0.5
ax2.plot(and_line, or_line, 'g-', linewidth=4, 
        label='Decision Line: OR = AND + 0.5', zorder=4)

# Shade regions
ax2.fill_between(and_line, -0.3, or_line, alpha=0.2, color='red', 
                label='Predict 0 (below line)')
ax2.fill_between(and_line, or_line, 1.5, alpha=0.2, color='blue', 
                label='Predict 1 (above line)')

for (and_val, or_val), label in zip(xor_transformed, xor_labels):
    color = 'red' if label == 0 else 'blue'
    marker = 'o' if label == 0 else 's'
    ax2.scatter(and_val, or_val, c=color, s=500, marker=marker, 
               alpha=0.7, edgecolors='k', linewidth=3, zorder=5)
    ax2.text(and_val + 0.08, or_val + 0.08, f'({and_val},{or_val})', 
            ha='left', va='bottom', fontsize=11, fontweight='bold')

ax2.set_xlabel('x1 AND x2', fontsize=14, fontweight='bold')
ax2.set_ylabel('x1 OR x2', fontsize=14, fontweight='bold')
ax2.set_title('Transformed Features (AND, OR)\n✓ Linearly Separable!', 
             fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-0.3, 1.3)
ax2.set_ylim(-0.3, 1.5)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("🎉 WITH THE RIGHT FEATURES, XOR IS LINEARLY SEPARABLE!")
print("="*70)
print("\nA single neuron CAN solve it if we give it (AND, OR) features.")
print("\nBut there's a problem...")
print("  • We had to MANUALLY engineer the features")
print("  • We needed to know that AND and OR would help")
print("  • For complex problems, what features should we use?")
print("\n💡 Solution: Let the network LEARN the features automatically!")
print("   That's exactly what hidden layers do...")
print("="*70)

### Exercise 12: Two Neurons Working Together

The solution: use **two neurons** in a hidden layer, each drawing a different line. Then combine their outputs with a third neuron.

**Architecture:**
```
Input (x1, x2)
     ↓
Hidden Layer: 2 neurons with ReLU
     ↓  
Output Layer: 1 neuron with sigmoid
```

Each hidden neuron learns a different pattern, and the output neuron combines them.

In [ ]:
# Let's visualize the architecture difference
Chapter4.visualize_network_architectures()

print("\n" + "="*70)
print("KEY DIFFERENCE")
print("="*70)
print("\nSingle Neuron (left):")
print("  • Input → Output (direct)")
print("  • Computes: y = w₁·x₁ + w₂·x₂ + b")
print("  • Decision boundary: ONE straight line")
print("  • Can't solve XOR!")
print("\n2-Layer Network (right):")
print("  • Input → Hidden Layer → Output")
print("  • Hidden layer: TWO neurons (h₁, h₂)")
print("  • Each hidden neuron draws its own line")
print("  • Output neuron combines them")
print("  • Can solve XOR! ✓")
print("\n💡 The hidden layer learns FEATURES (useful patterns)")
print("   The output layer combines features to make decisions")
print("="*70)

In [ ]:
def two_layer_network(x1, x2, W1, b1, W2, b2):
    """
    A 2-layer neural network.
    
    Args:
        x1, x2: inputs
        W1: weights for hidden layer (2x2 matrix)
        b1: biases for hidden layer (2 values)
        W2: weights for output layer (2 values)
        b2: bias for output layer (1 value)
    
    Returns:
        output, (h1, h2) - output and hidden layer activations
    """
    # Hidden layer
    h1 = relu(W1[0, 0] * x1 + W1[0, 1] * x2 + b1[0])
    h2 = relu(W1[1, 0] * x1 + W1[1, 1] * x2 + b1[1])
    
    # Output layer
    z = W2[0] * h1 + W2[1] * h2 + b2
    output = sigmoid(z)
    
    return output, (h1, h2)

# Hand-crafted weights that solve XOR
# Hidden neuron 1 (OR): Fires when h1 = relu(2*x1 + 2*x2 - 1)
#   - Activates when x1 + x2 >= 0.5
# Hidden neuron 2 (AND): Fires when h2 = relu(2*x1 + 2*x2 - 3)
#   - Activates when x1 + x2 >= 1.5 (both inputs are 1)
# Output: XOR = (x1 OR x2) AND NOT (x1 AND x2)
#   - When only OR fires: output is high (class 1)
#   - When both OR and AND fire: they cancel out, output is low (class 0)

W1 = np.array([
    [2.0, 2.0],   # OR neuron (fires when at least one input is 1)
    [2.0, 2.0]    # AND neuron (fires when both inputs are 1)
])
b1 = np.array([-1.0, -3.0])  # AND has higher threshold

W2 = np.array([2.0, -6.0])   # Positive OR, strong negative AND
b2 = -1.0

print("Hand-crafted 2-layer network on XOR:\n")
print(f"{'Input':<12} {'h1 (OR)':<10} {'h2 (AND)':<10} {'Output':<10} {'Predicted':<10} {'True':<10} {'Correct?'}")
print("="*80)

for (x1, x2), expected in zip(xor_inputs, xor_labels):
    output, (h1, h2) = two_layer_network(x1, x2, W1, b1, W2, b2)
    pred = 1 if output >= 0.5 else 0
    match = "✓" if pred == expected else "✗"
    print(f"({x1}, {x2}){' '*6} {h1:<10.2f} {h2:<10.2f} {output:<10.4f} {pred:<10} {expected:<10} {match}")

print("\n" + "="*80)
print("✓ Success! A 2-layer network CAN solve XOR.")
print("\nKey insight:")
print("  • Hidden neuron 1 learns: OR pattern (at least one input is 1)")
print("  • Hidden neuron 2 learns: AND pattern (both inputs are 1)")
print("  • Output combines them: XOR = OR AND NOT(AND)")
print("\nEach neuron learns a FEATURE. The output combines features.")
print("This is the KEY idea in neural networks!")
print("="*80)

### Exercise 13: NumPy for Efficiency

Before we train this network, let's switch to **NumPy** for faster computation.

Instead of writing loops, we'll use **matrix multiplication**.

In [ ]:
# TODO: Rewrite the network using NumPy matrix operations

def forward_pass_numpy(X, W1, b1, W2, b2):
    """
    Forward pass using NumPy (vectorized, fast).
    
    Args:
        X: input matrix (batch_size, 2)
        W1: hidden weights (2, 2)
        b1: hidden biases (2,)
        W2: output weights (2,)
        b2: output bias (scalar)
    
    Returns:
        predictions (batch_size, 1), hidden activations (batch_size, 2)
    """
    # Hidden layer: Z1 = X @ W1.T + b1, then ReLU
    Z1 = X @ W1.T + b1  # (batch_size, 2)
    H = np.maximum(0, Z1)  # ReLU
    
    # Output layer: Z2 = H @ W2 + b2, then sigmoid
    Z2 = H @ W2 + b2  # (batch_size,)
    predictions = sigmoid(Z2)  # (batch_size,)
    
    return predictions, H

# Test that it gives same results
print("Testing NumPy implementation:\n")
preds, hidden = forward_pass_numpy(xor_inputs, W1, b1, W2, b2)

for i, ((x1, x2), pred, expected) in enumerate(zip(xor_inputs, preds, xor_labels)):
    pred_class = 1 if pred >= 0.5 else 0
    match = "✓" if pred_class == expected else "✗"
    print(f"({x1}, {x2}) → {pred:.4f} (class {pred_class}) | True: {expected} {match}")

print("\n✓ NumPy version works! And it's much faster for large networks.")

### Exercise 14: Train the Network with Gradient Descent

Now the exciting part: start with **random weights** and train the network to solve XOR!

We'll use the same gradient descent approach, but now we have multiple weights to update.

In [ ]:
def compute_loss_network(X, y, W1, b1, W2, b2):
    """Compute average BCE loss for the network."""
    predictions, _ = forward_pass_numpy(X, W1, b1, W2, b2)
    losses = np.array([binary_cross_entropy(yt, yp) for yt, yp in zip(y, predictions)])
    return np.mean(losses)

def compute_gradients_numerical(X, y, W1, b1, W2, b2, h=1e-4):
    """
    Compute gradients for ALL parameters using numerical differentiation.
    This is slow but correct - we'll use backpropagation later for speed.
    """
    grad_W1 = np.zeros_like(W1)
    grad_b1 = np.zeros_like(b1)
    grad_W2 = np.zeros_like(W2)
    grad_b2 = 0.0
    
    # Gradient for W1
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            W1[i, j] += h
            loss_plus = compute_loss_network(X, y, W1, b1, W2, b2)
            W1[i, j] -= 2 * h
            loss_minus = compute_loss_network(X, y, W1, b1, W2, b2)
            W1[i, j] += h  # Restore
            grad_W1[i, j] = (loss_plus - loss_minus) / (2 * h)
    
    # Gradient for b1
    for i in range(b1.shape[0]):
        b1[i] += h
        loss_plus = compute_loss_network(X, y, W1, b1, W2, b2)
        b1[i] -= 2 * h
        loss_minus = compute_loss_network(X, y, W1, b1, W2, b2)
        b1[i] += h  # Restore
        grad_b1[i] = (loss_plus - loss_minus) / (2 * h)
    
    # Gradient for W2
    for i in range(W2.shape[0]):
        W2[i] += h
        loss_plus = compute_loss_network(X, y, W1, b1, W2, b2)
        W2[i] -= 2 * h
        loss_minus = compute_loss_network(X, y, W1, b1, W2, b2)
        W2[i] += h  # Restore
        grad_W2[i] = (loss_plus - loss_minus) / (2 * h)
    
    # Gradient for b2
    original_b2 = b2
    b2_plus = original_b2 + h
    loss_plus = compute_loss_network(X, y, W1, b1, W2, b2_plus)
    b2_minus = original_b2 - h
    loss_minus = compute_loss_network(X, y, W1, b1, W2, b2_minus)
    grad_b2 = (loss_plus - loss_minus) / (2 * h)
    
    return grad_W1, grad_b1, grad_W2, grad_b2

print("✓ Gradient computation ready.")
print("\nNote: Numerical gradients are slow but always correct.")
print("Later we'll learn 'backpropagation' which computes the same gradients much faster!")

In [ ]:
# TODO: Train the network on XOR

# Initialize random weights
np.random.seed(42)
W1 = np.random.randn(2, 2) * 0.5
b1 = np.random.randn(2) * 0.5
W2 = np.random.randn(2) * 0.5
b2 = np.random.randn() * 0.5

learning_rate = 1.0
num_steps = 1000

loss_history = []

print("Training 2-layer network on XOR...\n")
print(f"{'Step':<8} {'Loss':<12} {'Accuracy'}")
print("="*35)

for step in range(num_steps):
    # Forward pass
    predictions, _ = forward_pass_numpy(xor_inputs, W1, b1, W2, b2)
    
    # Compute loss
    loss = compute_loss_network(xor_inputs, xor_labels, W1, b1, W2, b2)
    loss_history.append(loss)
    
    # Compute accuracy
    pred_classes = (predictions >= 0.5).astype(int)
    accuracy = np.mean(pred_classes == xor_labels)
    
    # Print progress
    if step % 100 == 0 or step == num_steps - 1:
        print(f"{step:<8} {loss:<12.6f} {accuracy*100:.0f}%")
    
    # Compute gradients (this is the slow part)
    if step < num_steps - 1:  # Skip on last iteration
        grad_W1, grad_b1, grad_W2, grad_b2 = compute_gradients_numerical(
            xor_inputs, xor_labels, W1, b1, W2, b2
        )
        
        # Update weights
        W1 -= learning_rate * grad_W1
        b1 -= learning_rate * grad_b1
        W2 -= learning_rate * grad_W2
        b2 -= learning_rate * grad_b2

print("\n" + "="*35)
print("✓ Training complete!")
print("="*35)

In [ ]:
# Plot training progress
Chapter4.visualize_training_progress(loss_history, title='Training Progress: Loss Decreasing')

print(f"Initial loss: {loss_history[0]:.6f}")
print(f"Final loss:   {loss_history[-1]:.6f}")
print(f"Reduction:    {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")

In [ ]:
# Test final predictions
final_predictions, final_hidden = forward_pass_numpy(xor_inputs, W1, b1, W2, b2)

print("\nFinal Predictions:\n")
print(f"{'Input':<12} {'Prediction':<15} {'Class':<10} {'True':<10} {'Correct?'}")
print("="*60)

for (x1, x2), pred, true_label in zip(xor_inputs, final_predictions, xor_labels):
    pred_class = 1 if pred >= 0.5 else 0
    correct = "✓" if pred_class == true_label else "✗"
    print(f"({x1}, {x2}){' '*7} {pred:.4f}{' '*8} {pred_class:<10} {true_label:<10} {correct}")

accuracy = np.mean((final_predictions >= 0.5).astype(int) == xor_labels)
print(f"\nFinal Accuracy: {accuracy*100:.0f}%")

if accuracy == 1.0:
    print("\n" + "="*60)
    print("🎉 PERFECT! The network learned XOR from random initialization!")
    print("="*60)
    print("\nWhat just happened?")
    print("  1. Started with RANDOM weights (knew nothing)")
    print("  2. Computed how wrong we were (loss)")
    print("  3. Computed gradients (which way to improve)")
    print("  4. Updated all weights simultaneously")
    print("  5. Repeated 1000 times")
    print("  6. Network LEARNED the XOR pattern!")
    print("\nThis is the foundation of ALL neural network training.")
    print("="*60)
else:
    print("\nAlmost there! Try training for more steps or adjusting learning rate.")

### Exercise 15: What Did the Hidden Neurons Learn?

Let's see what patterns each hidden neuron detected.

In [ ]:
# Visualize the XOR network decision boundary in 2D
Chapter4.visualize_xor_decision_boundary(W1, b1, W2, b2, forward_pass_numpy, xor_inputs, xor_labels,
                                        title="XOR Network - Learned Decision Boundary")

print("\n" + "="*70)
print("XOR DECISION BOUNDARY")
print("="*70)
print("\nNotice:")
print("  • The decision boundary is now CURVED (not a straight line!)")
print("  • It correctly separates:")
print("    - Red points (0,0) and (1,1) in one region")
print("    - Blue points (0,1) and (1,0) in another region")
print("  • The green line shows where probability = 0.5")
print("  • Color intensity shows confidence (darker = more confident)")
print("\n✓ A 2-layer network can learn NON-LINEAR decision boundaries!")
print("="*70)

# Visualize what each hidden neuron detects
Chapter4.visualize_hidden_neuron_activations(W1, b1, xor_inputs, xor_labels)

print("\nEach hidden neuron learns to detect a different pattern:")
print("  • Neuron 1 detects one diagonal pattern")
print("  • Neuron 2 detects a complementary pattern")
print("  • The output neuron combines these to solve XOR!")

### Visualize the XOR Decision Boundary

Let's see the decision boundary the network learned in 2D space.

In [ ]:
print("Hidden Layer Activations:\n")
print(f"{'Input':<12} {'Hidden 1':<12} {'Hidden 2':<12} {'Pattern'}")
print("="*60)

for (x1, x2), (h1, h2) in zip(xor_inputs, final_hidden):
    pattern = ""
    if h1 > 0.5 and h2 < 0.5:
        pattern = "Detected: at least one input is 1, but not both"
    elif h1 < 0.5:
        pattern = "Detected: both inputs are 0"
    elif h2 > 0.5:
        pattern = "Detected: both inputs are 1"
    
    print(f"({x1}, {x2}){' '*7} {h1:<12.4f} {h2:<12.4f} {pattern}")

print("\n" + "="*60)
print("KEY INSIGHT: Hidden neurons learn FEATURES")
print("="*60)
print("\nEach neuron specializes in detecting a different pattern:")
print("  • We didn't tell them what patterns to detect")
print("  • They DISCOVERED useful features through training")
print("  • The output neuron combines these features")
print("\nThis is the fundamental idea behind deep learning:")
print("  → Learn representations (features) from data")
print("  → Not hand-coded rules!")
print("="*60)

---

## Summary of Part 1

### What You've Learned

1. **A neuron is just y = w×x + b** - a linear function

2. **Activation functions add non-linearity**
   - Sigmoid: squashes to (0,1), use for output probabilities
   - ReLU: sets negatives to 0, use for hidden layers

3. **Loss functions measure how wrong we are**
   - Binary Cross-Entropy (from Chapter 2!)
   - Lower loss = better predictions

4. **Gradient descent finds good weights automatically**
   - Gradient = slope of loss curve
   - Update rule: w_new = w_old - lr × gradient
   - Walk downhill to minimize loss

5. **Multiple layers enable complex patterns**
   - Single neuron: can only draw straight lines
   - Multiple neurons: can learn curved boundaries
   - Hidden layers learn **features** (useful patterns)

6. **Learning = discovering representations**
   - Networks learn what features to detect
   - Not hand-coded rules, but patterns found in data
   - This is the core idea of deep learning!
